In [ ]:
# V2 Agent Architecture Search Notebook
!pip install -q openai langgraph langchain-openai tavily-python tabulate

import os, time, textwrap
from openai import OpenAI
from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from google.colab import userdata
from IPython.display import display, Markdown
from tabulate import tabulate

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)
tavily = TavilyClient(api_key=TAVILY_API_KEY)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

QUESTION = "Compare AWS and Azure for GenAI workloads."

def call_llm(prompt):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )
    return r.choices[0].message.content, r.usage.total_tokens

def pretty_result(r):
    print("="*100)
    print(f"Architecture: {r['name']}")
    print(f"Quality: {r['quality']}/10 | Latency: {r['latency']:.2f}s | Tokens: {r['tokens']} | Score: {r['score']:.2f}")
    print("="*100)
    display(Markdown(r["answer"]))

def direct(q):
    s=time.time(); a,t=call_llm(q)
    return {"name":"DirectLLM","answer":a,"tokens":t,"latency":time.time()-s}

def plan(q):
    s=time.time()
    p,_=call_llm("Create a plan for: "+q)
    a,t=call_llm(f"Question:{q}\nPlan:{p}\nAnswer.")
    return {"name":"PlanSolve","answer":a,"tokens":t,"latency":time.time()-s}

def search(q):
    res=tavily.search(query=q,max_results=3)
    return "\n".join([x["content"][:400] for x in res["results"]])

def react(q):
    s=time.time()
    obs=search(q)
    a,t=call_llm(f"Question:{q}\nSearch:{obs}\nAnswer with citations.")
    return {"name":"ReAct","answer":a,"tokens":t,"latency":time.time()-s}

class State(TypedDict):
    question:str
    research:str
    answer:str

def research_node(state): return {"research": search(state["question"])}
def summarize_node(state):
    a,_=call_llm(f"Question:{state['question']}\nResearch:{state['research']}\nSummarize.")
    return {"answer":a}

g=StateGraph(State)
g.add_node("research", research_node)
g.add_node("summarize", summarize_node)
g.add_edge(START,"research"); g.add_edge("research","summarize"); g.add_edge("summarize", END)
graph=g.compile()

def graph_arch(q):
    s=time.time(); out=graph.invoke({"question":q})
    return {"name":"LangGraph","answer":out["answer"],"tokens":0,"latency":time.time()-s}

def judge(q,a):
    x,_=call_llm(f"Score 1-10 only. Question:{q}\nAnswer:{a}")
    try:return max(1,min(10,int(''.join(filter(str.isdigit,x))[:2])))
    except:return 5

architectures=[direct,plan,react,graph_arch]
results=[]
for fn in architectures:
    r=fn(QUESTION)
    r["quality"]=judge(QUESTION,r["answer"])
    r["score"]=0.8*r["quality"]-0.05*r["latency"]
    results.append(r)

results=sorted(results,key=lambda x:x["score"], reverse=True)

table=[[r["name"],r["quality"],round(r["latency"],2),r["tokens"],round(r["score"],2)] for r in results]
print(tabulate(table, headers=["Architecture","Quality","Latency","Tokens","Score"], tablefmt="github"))

for r in results:
    pretty_result(r)

print("\nWINNER:", results[0]["name"])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.2 MB/s eta 0:00:00
| Architecture   |   Quality |   Latency |   Tokens |   Score |
|----------------|-----------|-----------|----------|---------|
| DirectLLM      |         9 |     12.79 |      860 |    6.56 |
| ReAct          |         9 |     14.13 |     1153 |    6.49 |
| LangGraph      |         8 |      6.54 |        0 |    6.07 |
| PlanSolve      |         9 |     33.23 |     1996 |    5.54 |
Architecture: DirectLLM
Quality: 9/10 | Latency: 12.79s | Tokens: 860 | Score: 6.56


When comparing AWS (Amazon Web Services) and Azure (Microsoft Azure) for Generative AI workloads, several factors come into play, including services offered, scalability, integration, ecosystem, pricing, and user experience. Here’s a detailed comparison:

### 1. Core AI Services
**AWS:**
- **Amazon SageMaker:** Comprehensive platform for building, training, and deploying machine learning models. Supports multiple frameworks like TensorFlow, PyTorch, and MXNet.
- **AWS Lambda:** Ideal for serverless applications and can run code without provisioning servers, which is useful for ML workflows.
- **Amazon Comprehend, Rekognition, and Polly:** Various AI services for natural language processing, image recognition, and text-to-speech.
- **Foundation Models:** AWS provides access to models like GPT-3 via Amazon Bedrock, enhancing generative AI capabilities.

**Azure:**
- **Azure Machine Learning:** Similar to SageMaker, with strong support for framework interoperability and various deployment options.
- **Azure OpenAI Service:** Direct access to OpenAI models such as ChatGPT, useful for building conversational applications.
- **Cognitive Services:** Provides a range of pre-built models for language understanding, speech, vision, and decision-making tasks.
- **Azure Databricks:** Optimized Apache Spark platform for analytics and machine learning.

### 2. Scalability
**AWS:**
- Excellent scalability with auto-scaling features and a large number of instance types optimized for different workloads (GPU instances for training deep learning models).
- Global region and availability zone presence help in scaling applications globally.

**Azure:**
- Comparable scalability with Azure's Virtual Machines, Kubernetes Service, and Azure Functions.
- Regions and availability zones also support global deployments, often with lower latency thanks to strategic location distribution.

### 3. Integration with Other Services
**AWS:**
- Strong integration with other AWS services for data storage (S3), databases (RDS, DynamoDB), and developer tools (CodePipeline, CodeBuild).
- Supports infrastructure as code with AWS CloudFormation for deploying complex AI architectures.

**Azure:**
- Deep integration with Microsoft products (e.g., Microsoft 365, Dynamics 365) allows for seamless incorporation of AI capabilities into business applications.
- Strong support for Azure DevOps and infrastructure as code tools like ARM templates and Bicep.

### 4. Ecosystem and Marketplace
**AWS:**
- Extensive marketplace with numerous third-party AI and ML tools.
- Vibrant community and support structure including tutorials, forums, and extensive documentation.

**Azure:**
- Integration with GitHub enhances developers' workflows and allows easy collaboration.
- Marketplace also offers a broad range of AI tools but integrates more directly with Microsoft products.

### 5. Pricing Models
**AWS:**
- Pay-as-you-go model, but can become complex due to the number of services and resources.
- Savings plans and reserved instances can provide significant cost efficiencies for consistent workloads.

**Azure:**
- Similar pay-as-you-go model with potential discounts for long-term commitments.
- Azure has good pricing calculators that allow users to estimate costs based on their workload.

### 6. User Experience and Learning Curve
**AWS:**
- AWS Management Console is powerful but can be overwhelming for newcomers due to the vast array of options.
- AWS offers a range of training resources including the AWS Training and Certification program.

**Azure:**
- Azure Portal is generally considered more user-friendly, especially for those already familiar with Microsoft products.
- Microsoft Learn provides comprehensive learning paths for AI and machine learning, often seen as more structured than AWS's offerings.

### 7. Security and Compliance
Both platforms offer high levels of security and compliance options tailored to various industries, including healthcare, finance, and governmental organizations. Both services provide tools to manage identity and access, encryption, and auditing.

### Conclusion
Both AWS and Azure provide robust capabilities for generative AI workloads. The choice between the two often depends on specific business needs, existing technology stacks, and a company’s familiarity with each platform. AWS may appeal more to those needing a broad range of services and flexibility, while Azure could be preferable for organizations heavily invested in the Microsoft ecosystem.

Architecture: ReAct
Quality: 9/10 | Latency: 14.13s | Tokens: 1153 | Score: 6.49


When evaluating AWS and Azure for Generative AI (GenAI) workloads, several key factors come into play, including the range of services, ease of use, integration capabilities, and community support. Below is a comparison based on insights from various resources, including the K21Academy video detailing their strengths and weaknesses.

### Services and Tools

1. **AWS**:
   - **SageMaker**: AWS offers SageMaker, a comprehensive suite for building, training, and deploying machine learning models. SageMaker provides robust tooling, automated services, and integrates well with other AWS resources, making it powerful for GenAI workloads (K21Academy).
   - **Amazon Bedrock**: This new service allows users to build and scale generative AI applications more efficiently by providing access to multiple foundation models (Amazon, OpenAI, etc.) (K21Academy).

2. **Azure**:
   - **Azure AI and Azure OpenAI Service**: Azure integrates seamlessly with Microsoft's ecosystem, which can be an advantage for businesses already using Microsoft products. Azure AI Foundry provides developers with tools to build and integrate AI models easily. The Azure OpenAI Service allows access to powerful models such as ChatGPT and DALL-E, highlighting Microsoft’s strong partnership with OpenAI (K21Academy).
   - **Pre-trained Models**: Azure offers various pre-trained models directly through its AI services, which can accelerate development workflows (K21Academy).

### Community and Support

- **AWS**: AWS has a large and vibrant community, providing extensive documentation, tutorials, and forums for support. The AWS marketplace also offers numerous third-party solutions tailored to specific GenAI needs (AWS documentation).
  
- **Azure**: Similar to AWS, Azure has a robust support system. However, its focus on integration with Microsoft products may appeal more to enterprises already invested in other Microsoft technologies (K21Academy).

### Learning Curve

- **AWS**: Many users find AWS services to have a steeper learning curve due to the breadth of services and the complexity involved in using them effectively. However, once mastered, AWS can be highly capable for advanced use cases in Generative AI (K21Academy).
  
- **Azure**: Azure is often perceived as more user-friendly, especially for those accustomed to the Microsoft ecosystem. Its tools are designed to be intuitive and seamlessly integrate with services like Power BI and Dynamics (K21Academy).

### Vendor Lock-In

- **AWS**: There's a risk of vendor lock-in, particularly if you become heavily reliant on AWS-specific services and tools. While this can drive efficiency, it may also limit flexibility in the long term (K21Academy).
  
- **Azure**: As Azure becomes more intertwined with commonly used Microsoft services, there may also be concerns about vendor lock-in. However, Microsoft's push for open standards may mitigate this risk (K21Academy).

### Conclusion

Ultimately, the choice between AWS and Azure for Generative AI workloads depends on your specific needs and existing infrastructure. AWS may be better for organizations with a strong commitment to its ecosystem and those needing advanced tools like SageMaker and Bedrock. In contrast, Azure could be more appealing to enterprises that are heavily integrated into the Microsoft ecosystem and value ease of use and rapid deployment through pre-trained models. 

Both platforms offer robust solutions, but careful consideration of your organization's needs, existing tools, and future scalability will guide the best choice for your Generative AI initiatives. 

For detailed insights, you might want to watch the K21Academy video titled "GenAI on AWS vs Azure: Which One Should You Learn & Why?" for a more in-depth discussion on the topic.

Architecture: LangGraph
Quality: 8/10 | Latency: 6.54s | Tokens: 0 | Score: 6.07


When comparing AWS and Azure for Generative AI workloads, both platforms offer robust AI/ML services, yet they have distinct strengths that cater to different needs. 

**AWS Strengths:**
- **Amazon Bedrock & SageMaker:** These platforms provide comprehensive tools for building and deploying machine learning models, with SageMaker particularly strong in offering features for training and scaling models efficiently.
- **Enterprise Dominance:** AWS is often favored by organizations that already have a significant investment in AWS infrastructure, making integration and scaling smoother.

**Azure Strengths:**
- **Azure AI Foundry & Azure OpenAI:** Azure excels in its integration with OpenAI, allowing businesses to harness powerful generative AI models more seamlessly. It is particularly advantageous for enterprises invested in other Microsoft products, promoting a cohesive ecosystem.
- **Community Support & Learning Resources:** Despite some challenges in learning complexities, Azure benefits from strong community support and resources, particularly for users familiar with the Microsoft ecosystem.

**Considerations:**
- **Vendor Lock-In:** Both platforms have some degree of vendor lock-in, which might be a concern for organizations looking for flexibility.
- **Use Cases:** The choice may depend on specific use cases: AWS may be preferred for traditional enterprise applications, while Azure might be favored for applications closely tied with Microsoft products.

In summary, AWS might be more suitable for enterprises already integrated into its ecosystem, while Azure could be the optimal choice for those leveraging Microsoft technologies and looking for advanced generative AI capabilities.

Architecture: PlanSolve
Quality: 9/10 | Latency: 33.23s | Tokens: 1996 | Score: 5.54


### Comparison of AWS and Azure for Generative AI Workloads

#### 1. Introduction
Generative AI refers to AI systems capable of generating text, images, audio, and other data types based on prompts or existing data. Selecting the right cloud platform for deploying generative AI workloads is crucial for achieving efficiency, scalability, and adaptability. This report compares two leading cloud providers, Amazon Web Services (AWS) and Microsoft Azure, focusing on their capabilities in supporting generative AI workloads.

#### 2. Assessment Criteria
The following criteria are essential for comparing AWS and Azure:
- **Infrastructure and Technology**
- **AI and Machine Learning Services**
- **Storage Solutions**
- **Scalability and Flexibility**
- **Security and Compliance**
- **Cost Analysis**
- **Support and Documentation**

#### 3. Infrastructure and Technology
**Infrastructure**: Both AWS and Azure provide robust infrastructure. 

- **AWS**: Offers EC2 P-series for GPU computing tailored for ML workloads and Amazon SageMaker for streamlined model training and deployment.
- **Azure**: Features N-series VMs optimized for high-performance computing tasks and Azure Machine Learning for end-to-end ML lifecycle management.

**GPU and TPU options**: 
- **AWS**: Supports various NVIDIA GPUs (e.g., A100, V100) and even offers custom Inferentia chips for high-performance inference tasks.
- **Azure**: Also provides NVIDIA GPUs and has introduced support for AMD MI-series GPUs.

**Instance Types**: Both platforms offer flexible pricing with on-demand and reserved instances, allowing customers to tailor their usage based on workload requirements.

#### 4. AI and Machine Learning Services
Both platforms offer a rich set of tools for developing and deploying AI solutions.

- **AWS**: 
  - **SageMaker**: Simplifies the entire ML workflow, including data prep, model building, training, and deployment.
  - **Other Services**: Amazon Comprehend (NLP), Lex (chatbots), and Polly (text-to-speech).

- **Azure**: 
  - **Cognitive Services**: Pre-trained models for vision, speech, language, and decision-making tasks, making it easier for developers to integrate AI capabilities.
  - **Azure ML**: Comprehensive ML platform with support for various frameworks.

Both provide options for custom model training and deploying pre-trained models.

#### 5. Storage Solutions
**AWS**: 
- **S3** for object storage, **EBS** for block storage, and **Glacier** for archival storage. 
- Offers great integration with machine learning compute instances and excels in scalability.

**Azure**: 
- **Blob Storage** for scalable object storage, and **Disk Storage** for virtual machines.
- Azure’s storage solutions are also well-integrated with its AI and ML services, ensuring seamless data access.

#### 6. Scalability and Flexibility
Both AWS and Azure are designed to scale efficiently.

- **AWS**: Features comprehensive auto-scaling capabilities and supports serverless architecture through AWS Lambda, easing deployment and management.
- **Azure**: Offers similar auto-scaling options and Azure Functions, fostering seamless integration within Azure’s ecosystem.

#### 7. Security and Compliance
- **AWS**: Strong focus on security with services like IAM (Identity Access Management), data encryption solutions, and compliance with various industry standards including ISO, HIPAA, and GDPR.
  
- **Azure**: Comprehensive security features including Azure Active Directory and compliance with global standards.

Both platforms prioritize data privacy and security, fostering trust among enterprises.

#### 8. Cost Analysis
- **AWS Pricing**: Offers a pay-as-you-go model, with costs varying based on instance types, storage solutions, and data transfer.
  
- **Azure Pricing**: Also features a similar pricing model. Each platform offers a free tier with limited capabilities, which can be useful for initial testing.

Cost for generative AI workloads can vary significantly based on the workload’s specific requirements, instances used, and chosen services.

#### 9. Support and Documentation
- **AWS**: Extensive documentation, developer forums, and a variety of training resources, including AWS certifications.
  
- **Azure**: Similarly strong documentation, with additional support resources available through Microsoft Learn and a vibrant community contributing to forums.

#### 10. Case Studies and Use Cases
Both platforms have compelling success stories in generative AI. AWS is utilized in applications like content generation in media, while Azure supports creative industries through its cognitive services.

#### 11. Conclusion and Recommendations
In conclusion, both AWS and Azure offer robust solutions for generative AI workloads. 

- **AWS** is recommended for enterprises needing extensive AI and ML tooling with powerful compute capabilities.
- **Azure** excels in providing pre-trained AI services and an integrated environment for enterprises already using Microsoft products.

Businesses should assess their specific needs, including existing infrastructure, use cases, and team expertise, before making a selection.

#### 12. Appendices
Include a glossary of terms and additional resources for further learning.

### Timeline
- **Week 1**: Research infrastructure and technology, AI services.
- **Week 2**: Analyze storage solutions, scalability, and flexibility.
- **Week 3**: Investigate security, compliance, and cost analysis.
- **Week 4**: Collect case studies and finalize conclusions.
- **Week 5**: Draft the report and circulate for feedback.
- **Week 6**: Finalize and prepare the presentation of findings.

### Resources Needed
- Access to AWS and Azure demo accounts for hands-on testing.
- Industry reports and research papers on cloud AI capabilities.
- Budget for potential cloud usage during testing.
- Access to forums or communities for feedback.

### Outcome
A detailed comparative report will assist organizations in making informed decisions about which cloud provider best suits their generative AI workloads while serving as a broader resource on cloud AI services.


WINNER: DirectLLM
